# Member 1 — eBird ingestion, year by year (AviTrend PH)

Same job as the plan's *Member 1* section, but for yearly files (`2014_EBIRD`, `2015_EBIRD`, …) that **don't have a checklist ID**. Checklists are rebuilt from the other columns.

**How to use**
1. Run **Setup** once.
2. **Part A, once per year:** set `YEAR` and `LINK`, then run the Part A cells. Each year is saved to `data/interim/`. Repeat for every year, in any order. Re-running a year overwrites it.
3. **Part B, once at the end:** combines all saved years and writes the plan's outputs to `data/processed/`, then runs the checks.

| Term | Meaning | Stored as |
|---|---|---|
| Detection | A usable complete checklist reported the species | `detected = 1` |
| True absence | A usable complete checklist did **not** report the species | `detected = 0` |
| Observation gap | No usable checklist in that cell and month | no row; `is_gap = True` in the effort grid |
| Low-data cell | Fewer than 5 usable checklists in the cell over the whole period | `low_data = True` |

**How a checklist is rebuilt without its ID:** rows with the same date, start time, location, protocol, duration, distance and number of observers are one checklist. Shared group checklists (copies of one outing) are identical on all of these, so they merge into one automatically. If a `GROUP IDENTIFIER` column exists, it is used directly.

## Setup (run once)

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
try:
    import pyarrow  # noqa: F401
    HAS_PARQUET = True
except ImportError:
    HAS_PARQUET = False
    print("pyarrow not installed: files will be CSV instead of Parquet")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/NASA Hackathon")   # <- where outputs go (a folder you can write to)
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
OUT_DIR = PROJECT_ROOT / "data" / "processed"

# ---- plan settings (leave as is unless the plan changes) ----
DATE_START, DATE_END = "2014-01-01", "2025-12-31"
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 14.00, 15.30, 120.50, 121.60     # Greater Manila
CELL_DEG = 0.009                                                     # ~1 km cells, same as Member 2
PROTOCOLS = ["Stationary", "Traveling"]
DUR_MIN, DUR_MAX, DIST_MAX_KM, MAX_OBSERVERS = 5, 300, 5.0, 10
FOCUS_MIN_DETECTIONS, FOCUS_MIN_RATE, REQUIRE_TRAITS = 50, 0.01, True
LOW_DATA_MIN = 5

# Species masterlist (Avilight). Must be shared "Anyone with the link can view".
MASTERLIST_LINK = "https://docs.google.com/spreadsheets/d/1k4oiChJPhkroGhKxU7gX2ngxLTs6Z-9jSWiCBpUINCw/export?format=csv"
ML_NAME_COL, ML_TOL_COL, ML_MIG_COL = "Species Name", "Light Tolerance", "Migratory Status"

In [ ]:
# ---------- helpers (no need to edit) ----------
N_ROWS = int(np.ceil(round((LAT_MAX - LAT_MIN) / CELL_DEG, 6)))
N_COLS = int(np.ceil(round((LON_MAX - LON_MIN) / CELL_DEG, 6)))

def to_csv_link(link):
    # Google Sheet share link -> CSV export link; Drive file link -> direct download link
    if "docs.google.com/spreadsheets" in link and "export" not in link:
        return link.split("/edit")[0].split("?")[0].rstrip("/") + "/export?format=csv"
    if "drive.google.com/file/d/" in link:
        return "https://drive.google.com/uc?export=download&id=" + link.split("/file/d/")[1].split("/")[0]
    return link

def load_table(link):
    # Accepts a Google Sheet link, a Drive file link, or a file path (csv / xlsx / eBird txt)
    p = Path(link)
    if p.exists():
        if p.suffix.lower() in (".xlsx", ".xls"):
            return pd.read_excel(p, dtype=str)
        sep = "\t" if p.suffix.lower() == ".txt" else ","
        return pd.read_csv(p, sep=sep, dtype=str, low_memory=False, quoting=3 if sep == "\t" else 0)
    url = to_csv_link(link)
    try:
        df = pd.read_csv(url, dtype=str, low_memory=False)
        if df.shape[1] > 3:
            return df
    except Exception:
        pass
    return pd.read_excel(url, dtype=str)   # Drive-hosted .xlsx

def standardize(df):
    df = df.copy()
    df.columns = [str(c).strip().upper() for c in df.columns]
    if "PROTOCOL TYPE" in df.columns and "PROTOCOL NAME" not in df.columns:
        df = df.rename(columns={"PROTOCOL TYPE": "PROTOCOL NAME"})
    return df

REQUIRED = ["OBSERVATION DATE", "LATITUDE", "LONGITUDE", "ALL SPECIES REPORTED", "PROTOCOL NAME",
            "DURATION MINUTES", "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "COMMON NAME", "OBSERVATION COUNT"]
HELPFUL = ["TIME OBSERVATIONS STARTED", "LOCALITY ID", "GROUP IDENTIFIER", "SAMPLING EVENT IDENTIFIER",
           "CATEGORY", "SCIENTIFIC NAME", "APPROVED", "LOCALITY", "LOCALITY TYPE"]

def report_columns(df):
    missing = [c for c in REQUIRED if c not in df.columns]
    absent = [c for c in HELPFUL if c not in df.columns]
    print(f"rows: {len(df):,} | columns: {df.shape[1]}")
    print("missing REQUIRED columns:", missing or "none")
    print("missing helpful columns: ", absent or "none")
    if "TIME OBSERVATIONS STARTED" not in df.columns:
        print("WARNING: no start time -> two checklists at the same place, day and effort would merge. Note it as a limitation.")
    if "CATEGORY" not in df.columns:
        print("NOTE: no CATEGORY -> 'sp.', slash and hybrid entries are removed by name instead.")
    if missing:
        raise ValueError(f"Cannot continue without: {missing}. 'ALL SPECIES REPORTED' and effort columns are essential.")

def truthy(s):
    return s.astype(str).str.strip().str.upper().isin(["1", "1.0", "TRUE", "YES"])

def grid_rc(lat, lon):
    return (np.floor((np.asarray(lat, float) - LAT_MIN) / CELL_DEG).astype(int),
            np.floor((np.asarray(lon, float) - LON_MIN) / CELL_DEG).astype(int))

def cell_id_from_rc(row, col):
    return np.array([f"{int(r):03d}_{int(k):03d}" for r, k in zip(row, col)], dtype=object)

def save(df, folder, name):
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / (f"{name}.parquet" if HAS_PARQUET else f"{name}.csv")
    df.to_parquet(path, index=False) if HAS_PARQUET else df.to_csv(path, index=False)
    print(f"saved {path.name:32s}{len(df):>12,} rows")

def load(folder, name):
    pq, cs = folder / f"{name}.parquet", folder / f"{name}.csv"
    return pd.read_parquet(pq) if pq.exists() else pd.read_csv(cs)

def process_year(df, year):
    # Clean one year's observation rows into checklists + detections, save to data/interim/
    log = []
    def step(name, n):
        log.append({"year": year, "step": name, "checklists_remaining": int(n)}); print(f"{name:44s}{n:>10,}")

    d = df.copy()
    d["date"] = pd.to_datetime(d["OBSERVATION DATE"], errors="coerce")
    bad = d["date"].isna().mean()
    if bad > 0.01:
        raise ValueError(f"{bad:.1%} of dates could not be read - check the date format in this file")
    for c in ["LATITUDE", "LONGITUDE", "DURATION MINUTES", "EFFORT DISTANCE KM", "NUMBER OBSERVERS"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d[(d["date"].dt.year == year) & d["date"].between(DATE_START, DATE_END)]
    if "APPROVED" in d.columns:
        d = d[truthy(d["APPROVED"])]

    # coverage warning: was the file cropped to a smaller area?
    if d["LATITUDE"].min() > LAT_MIN + 0.1 or d["LATITUDE"].max() < LAT_MAX - 0.1 or \
       d["LONGITUDE"].min() > LON_MIN + 0.1 or d["LONGITUDE"].max() < LON_MAX - 0.1:
        warnings.warn(f"{year}: file covers lat {d['LATITUDE'].min():.2f}-{d['LATITUDE'].max():.2f}, "
                      f"lon {d['LONGITUDE'].min():.2f}-{d['LONGITUDE'].max():.2f}; it may not cover all of Greater Manila")
    d = d[d["LATITUDE"].between(LAT_MIN, LAT_MAX) & d["LONGITUDE"].between(LON_MIN, LON_MAX)]

    # ---- rebuild the checklist key ----
    d["distance_km"] = np.where(d["PROTOCOL NAME"] == "Stationary", 0.0, d["EFFORT DISTANCE KM"])
    if "TIME OBSERVATIONS STARTED" in d.columns:
        t = pd.to_datetime(d["TIME OBSERVATIONS STARTED"].astype(str), errors="coerce", format="mixed")
        d["hour_started"] = t.dt.hour + t.dt.minute / 60
        time_key = t.dt.strftime("%H:%M").fillna("")
    else:
        d["hour_started"] = np.nan
        time_key = pd.Series("", index=d.index)
    loc_key = d["LOCALITY ID"].fillna("") if "LOCALITY ID" in d.columns else \
        d["LATITUDE"].round(4).astype(str) + "," + d["LONGITUDE"].round(4).astype(str)
    key = (d["date"].dt.strftime("%Y-%m-%d") + "|" + time_key + "|" + loc_key + "|" + d["PROTOCOL NAME"].fillna("") + "|" +
           d["DURATION MINUTES"].round(0).astype(str) + "|" + d["distance_km"].round(2).astype(str) + "|" +
           d["NUMBER OBSERVERS"].astype(str))
    if "GROUP IDENTIFIER" in d.columns:
        g = d["GROUP IDENTIFIER"].fillna("").str.strip()
        key = np.where(g != "", "G|" + g, key)
    if "SAMPLING EVENT IDENTIFIER" in d.columns and "GROUP IDENTIFIER" not in d.columns:
        warnings.warn("SAMPLING EVENT IDENTIFIER found but no GROUP IDENTIFIER: group copies are merged by effort columns")
    hashes = pd.util.hash_pandas_object(pd.Series(key, index=d.index).astype(str), index=False).to_numpy()
    d["checklist_id"] = ["C%016x" % int(h) for h in hashes]   # plain Python strings: works on every pandas version
    d["complete"] = truthy(d["ALL SPECIES REPORTED"])

    # ---- one row per checklist (complete if any copy was complete) ----
    first = d.drop_duplicates("checklist_id")
    c = first[["checklist_id", "date", "hour_started", "PROTOCOL NAME", "DURATION MINUTES", "distance_km",
               "NUMBER OBSERVERS", "LATITUDE", "LONGITUDE"] +
              [x for x in ["LOCALITY", "LOCALITY ID", "LOCALITY TYPE"] if x in d.columns]].copy()
    c = c.merge(d.groupby("checklist_id")["complete"].max().reset_index(), on="checklist_id")
    step("checklists in Greater Manila (rebuilt)", len(c))
    c = c[c["complete"]];                                              step("complete checklists", len(c))
    c = c[c["PROTOCOL NAME"].isin(PROTOCOLS)];                         step("Stationary or Traveling", len(c))
    c = c[c["DURATION MINUTES"].between(DUR_MIN, DUR_MAX)];            step(f"duration {DUR_MIN}-{DUR_MAX} min", len(c))
    c = c[c["distance_km"] <= DIST_MAX_KM];                            step(f"distance <= {DIST_MAX_KM:g} km", len(c))
    c = c[c["NUMBER OBSERVERS"].between(1, MAX_OBSERVERS)];            step(f"observers <= {MAX_OBSERVERS}", len(c))

    # ---- detections: one row per checklist x species ----
    o = d[d["checklist_id"].isin(c["checklist_id"])].copy()
    if "CATEGORY" in o.columns:
        o = o[o["CATEGORY"].isin(["species", "issf", "form", "domestic"])]
    else:
        nm = o["COMMON NAME"].astype(str)
        o = o[~(nm.str.contains(r" sp\.|/|\bx\b| hybrid", regex=True))]
    o["count_num"] = pd.to_numeric(o["OBSERVATION COUNT"], errors="coerce")      # 'X' -> NaN
    sci = "SCIENTIFIC NAME" if "SCIENTIFIC NAME" in o.columns else "COMMON NAME"
    det = (o.groupby(["checklist_id", "COMMON NAME"], as_index=False)
             .agg(count_max=("count_num", "max"), any_x=("count_num", lambda s: s.isna().any()),
                  scientific_name=(sci, "first"))
             .rename(columns={"COMMON NAME": "species", "count_max": "count"}))
    det.loc[det["any_x"], "count"] = np.nan        # group copies: max count, not sum (avoids double counting)
    det = det.drop(columns="any_x")
    c = c[c["checklist_id"].isin(det["checklist_id"])];                step("has at least one bird record (final)", len(c))

    # ---- derived columns ----
    c["year"], c["month"] = c["date"].dt.year, c["date"].dt.month
    c["week"] = ((c["date"].dt.dayofyear - 1) // 7 + 1).clip(upper=52)
    r, k = grid_rc(c["LATITUDE"], c["LONGITUDE"])
    c["cell_row"], c["cell_col"], c["cell_id"] = r, k, cell_id_from_rc(r, k)
    c["within_2km"] = c["distance_km"] <= 2.0
    c = c.rename(columns={"PROTOCOL NAME": "protocol", "DURATION MINUTES": "duration_min", "NUMBER OBSERVERS": "n_observers",
                          "LATITUDE": "lat", "LONGITUDE": "lon", "LOCALITY": "locality", "LOCALITY ID": "locality_id",
                          "LOCALITY TYPE": "locality_type"}).drop(columns="complete")

    # ---- spot check: 3 most common species, recounted from the raw rows ----
    final_ids = set(c["checklist_id"])
    for s in det["species"].value_counts().head(3).index:
        raw = d.loc[(d["COMMON NAME"] == s) & d["checklist_id"].isin(final_ids), "checklist_id"].nunique()
        pipe = det.loc[(det["species"] == s) & det["checklist_id"].isin(final_ids), "checklist_id"].nunique()
        print(f"[{'PASS' if raw == pipe else 'FAIL'}] spot check {s}: pipeline {pipe:,} vs raw {raw:,}")

    det = det[det["checklist_id"].isin(final_ids)]
    det = (det.groupby(["checklist_id", "species"], as_index=False)
              .agg(count=("count", "max"), scientific_name=("scientific_name", "first")))
    assert c["checklist_id"].is_unique and not det.duplicated(["checklist_id", "species"]).any()
    save(c, INTERIM_DIR, f"checklists_{year}")
    save(det, INTERIM_DIR, f"detections_raw_{year}")
    pd.DataFrame(log).to_csv(INTERIM_DIR / f"filter_log_{year}.csv", index=False)
    return pd.DataFrame(log)

print(f"setup done | grid {N_ROWS} x {N_COLS} = {N_ROWS * N_COLS:,} cells | outputs -> {PROJECT_ROOT}")

## Part A — one year at a time

Change `YEAR` and `LINK`, then run the three cells below. Repeat for every year.

In [ ]:
YEAR = 2014
LINK = "PASTE_THE_SHARE_LINK_OF_2014_EBIRD_HERE"     # Google Sheet link, Drive file link, or file path

df = load_table(LINK)
df = standardize(df)
df.head()

In [ ]:
report_columns(df)     # stops here with a clear message if an essential column is missing

In [ ]:
year_log = process_year(df, YEAR)
year_log

## Part B — final export (run once, after all years are saved)

Combines every year in `data/interim/`, picks focus species, fills in the true absences, builds the effort grid, and writes the plan's outputs.

In [ ]:
years_found = sorted({int(p.stem.split("_")[-1]) for p in INTERIM_DIR.glob("checklists_*.*") if p.stem.split("_")[-1].isdigit()})
y0, y1 = int(DATE_START[:4]), int(DATE_END[:4])
years_done = [y for y in years_found if y0 <= y <= y1]          # only years inside the analysis window
skipped = sorted(set(years_found) - set(years_done))
print("years used:", years_done, "| skipped (outside window):", skipped or "none")
checklists = pd.concat([load(INTERIM_DIR, f"checklists_{y}") for y in years_done], ignore_index=True)
empty_years = sorted(set(years_done) - set(checklists["year"].unique()))
if empty_years:
    print("WARNING: these years have no usable checklists:", empty_years)
det = pd.concat([load(INTERIM_DIR, f"detections_raw_{y}") for y in years_done], ignore_index=True)
checklists["date"] = pd.to_datetime(checklists["date"])
# ---- safety: one row per checklist, one row per checklist x species ----
n_dup_chk = checklists["checklist_id"].duplicated().sum()
if n_dup_chk:
    print(f"NOTE: {n_dup_chk} checklist IDs appear more than once across years -> kept once")
checklists = checklists.drop_duplicates("checklist_id")
det = det[det["checklist_id"].isin(checklists["checklist_id"])]
dup_pairs = det[det.duplicated(["checklist_id", "species"], keep=False)]
if len(dup_pairs):
    print(f"NOTE: {len(dup_pairs)} duplicated checklist x species rows -> merged (max count). Examples:")
    print(dup_pairs.sort_values(["checklist_id", "species"]).head(10).to_string(index=False))
    det = (det.groupby(["checklist_id", "species"], as_index=False)
              .agg(count=("count", "max"), scientific_name=("scientific_name", "first")))
n_chk = len(checklists)
print(f"usable checklists: {n_chk:,} | detections: {len(det):,}")

# ---- species list + masterlist traits ----
species = (det.groupby("species", as_index=False)
              .agg(scientific_name=("scientific_name", "first"), n_detections=("checklist_id", "nunique")))
species["reporting_rate"] = species["n_detections"] / n_chk
def norm(s): return s.astype(str).str.strip().str.lower().str.replace(r"\s+", " ", regex=True).str.replace("’", "'")
if MASTERLIST_LINK:
    ml = pd.read_csv(to_csv_link(MASTERLIST_LINK))[[ML_NAME_COL, ML_TOL_COL, ML_MIG_COL]].dropna(subset=[ML_NAME_COL])
    ml["key"] = norm(ml[ML_NAME_COL])
    ml = ml.drop_duplicates("key").rename(columns={ML_TOL_COL: "light_tolerance", ML_MIG_COL: "migratory_status"})
    species["key"] = norm(species["species"])
    species = species.merge(ml[["key", "light_tolerance", "migratory_status"]], on="key", how="left").drop(columns="key")
else:
    species["light_tolerance"], species["migratory_status"] = np.nan, np.nan

threshold = max(FOCUS_MIN_DETECTIONS, int(np.ceil(FOCUS_MIN_RATE * n_chk)))
common = species["n_detections"] >= threshold
has_trait = species["light_tolerance"].notna()
species["focus"] = (common & has_trait) if REQUIRE_TRAITS else common
OUT_DIR.mkdir(parents=True, exist_ok=True)
species[common & ~has_trait][["species", "scientific_name", "n_detections"]] \
    .sort_values("n_detections", ascending=False).to_csv(OUT_DIR / "species_unmatched.csv", index=False)
focus_list = species.loc[species["focus"], "species"].sort_values().tolist()
print(f"species: {len(species):,} | threshold >= {threshold:,} checklists | common: {common.sum()} | "
      f"with trait: {(common & has_trait).sum()} | focus: {len(focus_list)}")
print("common species without a trait -> species_unmatched.csv (fix names in the masterlist)")

In [ ]:
# ---- zero-fill: every usable checklist x every focus species ----
idx = pd.MultiIndex.from_product([checklists["checklist_id"].values, focus_list], names=["checklist_id", "species"])
detections = (pd.DataFrame(index=idx).reset_index()
                .merge(det[det["species"].isin(focus_list)][["checklist_id", "species", "count"]],
                       on=["checklist_id", "species"], how="left", indicator=True))
detections["detected"] = (detections.pop("_merge") == "both").astype("int8")
detections["species"] = detections["species"].astype("category")
detections = detections[["checklist_id", "species", "detected", "count"]]

# ---- effort grid: every cell x month of the processed years ----
months = pd.period_range(f"{min(years_done)}-01", f"{max(years_done)}-12", freq="M")
rr, cc = np.meshgrid(np.arange(N_ROWS), np.arange(N_COLS), indexing="ij")
cells = pd.DataFrame({"cell_row": rr.ravel(), "cell_col": cc.ravel()})
cells["cell_id"] = cell_id_from_rc(cells["cell_row"].values, cells["cell_col"].values)
effort = cells.merge(pd.DataFrame({"year": months.year, "month": months.month}), how="cross")
counts = (checklists.assign(hours=checklists["duration_min"] / 60)
            .groupby(["cell_id", "year", "month"], as_index=False)
            .agg(n_checklists=("checklist_id", "size"), total_hours=("hours", "sum")))
effort = effort.merge(counts, on=["cell_id", "year", "month"], how="left")
effort[["n_checklists", "total_hours"]] = effort[["n_checklists", "total_hours"]].fillna(0)
effort["n_checklists"] = effort["n_checklists"].astype(int)
effort["is_gap"] = effort["n_checklists"] == 0
cell_tot = effort.groupby("cell_id")["n_checklists"].transform("sum")
effort["low_data"] = (cell_tot > 0) & (cell_tot < LOW_DATA_MIN)
print(f"detections: {len(detections):,} rows | effort grid: {len(effort):,} rows | gap share: {effort['is_gap'].mean():.1%}")

In [ ]:
# ---- save the plan's outputs ----
checklists_out = checklists[["checklist_id", "date", "year", "month", "week", "hour_started", "protocol", "duration_min",
                             "distance_km", "within_2km", "n_observers", "lat", "lon", "cell_row", "cell_col", "cell_id"] +
                            [x for x in ["locality", "locality_id", "locality_type"] if x in checklists.columns]] \
                 .sort_values(["date", "checklist_id"]).reset_index(drop=True)
species_out = species[["species", "scientific_name", "n_detections", "reporting_rate",
                       "light_tolerance", "migratory_status", "focus"]].sort_values("n_detections", ascending=False)
save(checklists_out, OUT_DIR, "checklists")
save(detections, OUT_DIR, "detections")
save(species_out, OUT_DIR, "species")
save(effort, OUT_DIR, "effort_grid")
filter_log = pd.concat([pd.read_csv(INTERIM_DIR / f"filter_log_{y}.csv") for y in years_done], ignore_index=True)
filter_log.to_csv(OUT_DIR / "filter_log.csv", index=False)
print("saved filter_log.csv")

### Checks (all must pass before handing off)

In [ ]:
results = []
def check(name, ok, detail=""):
    results.append(bool(ok)); print(f"[{'PASS' if ok else 'FAIL'}] {name} {detail}")

check("checklist_id is unique", checklists_out["checklist_id"].is_unique)
check("detections = checklists x focus species", len(detections) == n_chk * len(focus_list),
      f"({len(detections):,} vs {n_chk:,} x {len(focus_list)})")
check("filters hold", checklists_out["duration_min"].between(DUR_MIN, DUR_MAX).all()
      and (checklists_out["distance_km"] <= DIST_MAX_KM).all() and (checklists_out["n_observers"] <= MAX_OBSERVERS).all())
check("all inside Greater Manila", checklists_out["lat"].between(LAT_MIN, LAT_MAX).all()
      and checklists_out["lon"].between(LON_MIN, LON_MAX).all())
check("effort grid = cells x months", len(effort) == N_ROWS * N_COLS * len(months))
check("effort total = checklists", effort["n_checklists"].sum() == n_chk)
check("UP Diliman cell is 072_062", cell_id_from_rc(*grid_rc([14.655], [121.065]))[0] == "072_062")
missing_years = sorted(set(range(min(years_done), max(years_done) + 1)) - set(years_done))
check("no missing years in between", not missing_years, missing_years or "")
print("\nchecklists per year:"); print(checklists_out.groupby("year").size().to_string())
print("\nmost reported species (should be common urban birds):")
print(species_out.head(8)[["species", "n_detections", "reporting_rate", "focus"]].to_string(index=False))

In [ ]:
tot = effort.groupby(["cell_row", "cell_col"])["n_checklists"].sum().unstack() \
            .reindex(index=range(N_ROWS), columns=range(N_COLS)).fillna(0).values
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(7, 8))
    im = ax.imshow(np.where(tot > 0, np.log10(tot), np.nan), origin="lower", cmap="viridis",
                   extent=[LON_MIN, LON_MIN + N_COLS * CELL_DEG, LAT_MIN, LAT_MIN + N_ROWS * CELL_DEG])
    ax.set_facecolor("#e6e6e6"); fig.colorbar(im, ax=ax, label="log10(usable checklists per 1 km cell)")
    ax.set_title(f"eBird effort {min(years_done)}-{max(years_done)} (grey = observation gap)")
    (OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
    fig.savefig(OUT_DIR / "figures" / "hotspots.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"Checks: {sum(results)}/{len(results)} passed | years {years_done} | checklists {n_chk:,} "
      f"(<= 2 km: {int(checklists_out['within_2km'].sum()):,}) | focus species {len(focus_list)} | "
      f"cells with data {(tot > 0).sum():,} of {N_ROWS * N_COLS:,}")